# Week 1: Linear Regression with Regularization
## Sales Prediction with Price and Lag Features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Load and Prepare Data

In [ ]:
# Load H&M sales data
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)

# Select a single product category
product = df['name'].unique()[0]
df_product = df[df['name'] == product].copy()

print(f"Product: {product}")
print(f"Samples: {len(df_product)}")
df_product.head()

## Feature Engineering

In [ ]:
# Create lag features
df_product['lag_m1'] = df_product.groupby('id')['sales'].shift(1)
df_product['lag_m2'] = df_product.groupby('id')['sales'].shift(2)

# Create moving average
df_product['ma_3'] = df_product.groupby('id')['sales'].shift(1).rolling(3, min_periods=1).mean().values

# Price change features
df_product['price_change'] = df_product.groupby('id')['price'].diff()

# Fill NaN values
df_product[['lag_m1', 'lag_m2', 'ma_3', 'price_change']] = df_product[['lag_m1', 'lag_m2', 'ma_3', 'price_change']].fillna(0)

df_product[['sales', 'price', 'lag_m1', 'lag_m2', 'ma_3']].describe()

## Train/Test Split

In [ ]:
# Define features
features = ['price', 'lag_m1', 'lag_m2', 'ma_3', 'price_change']
target = 'sales'

# Split by time (last 20% as test)
split_idx = int(len(df_product) * 0.8)
train = df_product.iloc[:split_idx]
test = df_product.iloc[split_idx:]

X_train, y_train = train[features].values, train[target].values
X_test, y_test = test[features].values, test[target].values

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(train)}, Test: {len(test)}")

## Model Comparison

In [ ]:
models = {
    'OLS': LinearRegression(),
    'Lasso': Lasso(alpha=1.0, random_state=42),
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

results_df = pd.DataFrame(results)
results_df

## Coefficient Comparison

In [ ]:
# Extract coefficients for each model
coef_df = pd.DataFrame({
    'Feature': features,
    'OLS': models['OLS'].coef_,
    'Lasso': models['Lasso'].coef_,
    'Ridge': models['Ridge'].coef_,
    'ElasticNet': models['ElasticNet'].coef_
})

print("\nCoefficient Comparison:")
print(coef_df.round(3))

# Plot coefficients
fig, ax = plt.subplots(figsize=(10, 5))
coef_df.set_index('Feature')[['OLS', 'Lasso', 'Ridge', 'ElasticNet']].plot(kind='bar', ax=ax)
plt.title('Coefficient Comparison Across Models')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model')
plt.tight_layout()
plt.show()

## Regularization Effect (Alpha Tuning)

In [ ]:
# Test different alpha values for Lasso
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
lasso_results = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, random_state=42)
    lasso.fit(X_train_scaled, y_train)
    y_pred = lasso.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    lasso_results.append({
        'Alpha': alpha,
        'MAE': mae,
        'Non-zero Coefs': n_nonzero
    })

lasso_df = pd.DataFrame(lasso_results)
print("\nLasso Alpha Tuning:")
print(lasso_df)

# Plot alpha vs MAE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(lasso_df['Alpha'], lasso_df['MAE'], marker='o')
ax1.set_xscale('log')
ax1.set_xlabel('Alpha (Regularization Strength)')
ax1.set_ylabel('MAE')
ax1.set_title('Lasso: Alpha vs MAE')
ax1.grid(True, alpha=0.3)

ax2.plot(lasso_df['Alpha'], lasso_df['Non-zero Coefs'], marker='o', color='green')
ax2.set_xscale('log')
ax2.set_xlabel('Alpha (Regularization Strength)')
ax2.set_ylabel('Number of Non-zero Coefficients')
ax2.set_title('Lasso: Feature Selection Effect')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Predictions vs Actuals

In [ ]:
# Compare predictions for best model (Lasso)
best_model = models['Lasso']
y_pred = best_model.predict(X_test_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Sales')
ax.set_ylabel('Predicted Sales')
ax.set_title('Lasso: Predictions vs Actuals')
plt.tight_layout()
plt.show()